# PostgreSQL Quickstart

This guide will get you started with **Ormophine** for PostgreSQL. If you have used the SQLite version, the API is nearly identical, but adapted for PostgreSQL features and `psycopg`.

> **Note:**
> - Tables and columns become **Python attributes automatically** when you connect.
> - All queries use `%s` placeholders (psycopg style) and are parametrised safely.
> - The driver manages a **connection pool** for efficiency.

## Import the ORM

Everything you need is in `Ormophine.PostgreSQL`:

In [ ]:
from Ormophine.Postgresql import Driver, TableStructure, DataTypes, Join

## Connecting to a Database

Create a `Driver` instance with your PostgreSQL credentials:

In [ ]:
db = Driver(
    host="localhost",
    port=5432,
    username="postgres",
    password="your_password",
    db_name="mydb",
    pool_size=5,        # number of connections in the pool
    create_new_db=True           
)
# All existing tables are now available as attributes of `db`.
# For example, if a table 'users' already exists, you can use `db.users`.

> **Important:**
> The Driver **automatically discovers** every table in your default schema (`current_schema()`) and makes them available as attributes of the `db` object. Likewise, every column of those tables becomes an attribute of the corresponding `Table` object. No manual model definition is required.

## Creating Your First Table

Use `TableStructure` to define a new table, then call `db.create_table()`.

In [ ]:
# Define a 'users' table
users_schema = TableStructure("users")

users_schema.add_column(
    "id", DataTypes.SERIAL(),
    primary_key=True          # automatically NOT NULL and auto‑increment
)
users_schema.add_column(
    "username", DataTypes.VARCHAR(50),
    unique=True, not_null=True
)
users_schema.add_column(
    "age", DataTypes.INTEGER(),
    default_value=18
)

# Create the table – returns a Table object.
users = db.create_table(users_schema)
# Now `db.users` also points to the same object.

> **Tip:**
> `DataTypes.SERIAL()` is a convenient shortcut for an auto‑incrementing primary key. Other PostgreSQL‑specific types like `BOOLEAN()`, `JSONB()`, `UUID()` are all supported.

## Inserting Data

Single insert:

In [ ]:
users = db.users
users.insert({
    users.username: "alice",
    users.age: 25
})
# 'id' is automatically generated by the SERIAL column

Bulk insert:

In [ ]:
users.bulk_insert(
    columns=[users.username, users.age],
    data_list=[
        ("bob", 30),
        ("charlie", 22),
        ("diana", 28),
    ]
)

## Querying Data

Use `get_row()` to fetch rows. Select specific columns, filter with a `WHERE` condition, and order the results.

In [ ]:
# Fetch usernames and ages, ordered by age
all_users = users.get_row(
    which_columns=[users.username, users.age],
    order_by=users.age
)
for row in all_users:
    print(row)  # e.g., ('charlie', 22), ('alice', 25), ...

Building conditions with `ColumnsOperation`:

In [ ]:
# Users older than 24
condition = users.age > 24
older_users = users.get_row(
    which_columns=[users.username],
    where=condition
)
for (username,) in older_users:
    print(username)  # alice, bob, diana

## Using the `In()` Method for Advanced Filtering

The `In()` method on a `Column` or `ColumnsOperation` allows you to filter rows based on a list of values or a subquery. 
It supports two modes:
1. **Literal list (`data_list`)**: Pass a list of values directly.
2. **Subquery (`which_columns` and `where`)**: Build a `SELECT ... FROM ... WHERE ...` subquery by passing a single column (and an optional condition), similar to how `get_row()` works.

In [ ]:
# --- 1. Using In() with a direct list of values (data_list) ---
# Select users whose username is either 'alice' or 'bob'
condition_list = users.username.In(data_list=["alice", "bob"])

users_in_list = users.get_row(
    which_columns=[users.username, users.age],
    where=condition_list
)

for row in users_in_list:
    print(row)
# Output:
# ('alice', 25)
# ('bob', 30)

('alice', 25)
('bob', 30)


2. **Subquery mode**: Find users whose username exists in the `admins` table, excluding certain admins.

In [ ]:
# First, let's create a quick 'admins' table to test the subquery against
if not hasattr(db, 'admins'):
    admins_schema = TableStructure("admins")
    admins_schema.add_column("id", DataTypes.SERIAL(), primary_key=True)
    admins_schema.add_column("username", DataTypes.VARCHAR(50))
    db.create_table(admins_schema)

admins = db.admins
admins.bulk_insert(
    columns=[admins.username],
    data_list=[
        ("bob",),
        ("diana",),
        ("charlie",) # Note: charlie is still in users at this point
    ]
)

In [ ]:
# Find users whose username exists in the admins table,
# but only select admins where their username is not 'charlie'
condition_subquery = users.username.In(
    column=admins.username,
    where=admins.username != "charlie"
)

admin_users = users.get_row(
    which_columns=[users.username, users.age],
    where=condition_subquery,
    order_by=users.age
)

print("Users who are admins (excluding 'charlie' from subquery):")
for row in admin_users:
    print(row)
# Output:
# Users who are admins (excluding 'charlie' from subquery):
# ('diana', 28)
# ('bob', 30)

Users who are admins (excluding 'charlie' from subquery):
('diana', 28)
('bob', 30)


## Updating Data

Update rows that match a condition. You can use constant values or column‑based arithmetic.

In [ ]:
# Increase age by 1 for all users under 30
condition = users.age < 30
users.update(
    update={users.age: users.age + 1},   # ColumnsOperation
    where=condition
)

## Deleting Data

In [ ]:
# Remove user 'charlie'
condition = users.username == "charlie"
users.delete_row(where=condition)

## Complex Example with ColumnsOperation

Now let’s work with a `products` table and perform arithmetic and string manipulations.

1. **Create the products table**

In [ ]:
products_schema = TableStructure("products")
products_schema.add_column("id", DataTypes.SERIAL(), primary_key=True)
products_schema.add_column("name", DataTypes.TEXT())
products_schema.add_column("price", DataTypes.NUMERIC(10, 2))
products_schema.add_column("discount", DataTypes.NUMERIC(3, 2))  # e.g., 0.10
products_schema.add_column("category", DataTypes.VARCHAR(50))

db.create_table(products_schema)

2. **Insert sample data**

In [ ]:
products = db.products
products.bulk_insert(
    columns=[products.name, products.price, products.discount, products.category],
    data_list=[
        ("Widget", 19.99, 0.1, "gadgets"),
        ("Gadget Pro", 49.99, 0.2, "gadgets"),
        ("SuperTool", 29.99, 0.0, "tools"),
        ("MegaWidget", 99.99, 0.25, "gadgets"),
    ]
)

3. **Arithmetic expression** – compute final price and filter

In [ ]:
# final_price = price * (1 - discount)
final_price = products.price * (1 - products.discount)
condition = final_price > 30

result = products.get_row(
    which_columns=[products.name, final_price],
    where=condition
)

for name, price in result:
    print(f"{name}: ${price:.2f}")
# Output: Gadget Pro: $39.99, MegaWidget: $74.99

4. **String operations** – case‑insensitive search (PostgreSQL ILIKE is not used directly; we use `LOWER()` and `LIKE`)

In [ ]:
# Gadgets whose name contains "widget" (case‑insensitive)
condition = (products.category == "gadgets") & (products.name.lower().contains("widget"))

gadget_widgets = products.get_row(
    which_columns=[products.name, products.price],
    where=condition
)
for name, price in gadget_widgets:
    print(f"{name}: ${price}")
# Widget: $19.99, MegaWidget: $99.99

5. **Update using a calculation** – apply extra discount

In [ ]:
# Increase discount by 5 percentage points where discount < 20%
new_discount = products.discount + 0.05
products.update(
    update={products.discount: new_discount},
    where=products.discount < 0.2
)

6. **Deleting with a string condition**

In [ ]:
# Remove all tools
products.delete_row(where=products.category == "tools")

## Batch Operations – Transactions

For multiple statements that must run atomically, use `batch()`:

In [ ]:
batch = products.batch()
batch.insert({products.name: "Hammer", products.price: 12.50, products.discount: 0.0, products.category: "tools"})
batch.update(
    update={products.price: products.price * 1.1},  # 10% price increase
    where=products.discount == 0.0
)
batch.run()   # both operations execute in a single transaction

## Joining Tables

Assume we have an `orders` table referencing `products`.

In [ ]:
if not hasattr(db, 'orders'):
    orders_schema = TableStructure("orders")
    orders_schema.add_column("id", DataTypes.SERIAL(), primary_key=True)
    orders_schema.add_column("product_id", DataTypes.INTEGER())
    orders_schema.add_column("quantity", DataTypes.INTEGER())
    db.create_table(orders_schema)

orders = db.orders
orders.insert({orders.product_id: 1, orders.quantity: 3})
orders.insert({orders.product_id: 2, orders.quantity: 1})
orders.insert({orders.product_id: 4, orders.quantity: 5})

Now join `orders` with `products`:

In [ ]:
join_condition = orders.product_id == products.id

columns = [
    orders.id,
    products.name,
    orders.quantity,
    products.price * orders.quantity   # total cost
]

result = orders.join(
    columns=columns,
    joins_list=[Join.Inner(products, join_condition)],
    order_by=orders.id
)

for row in result:
    print(row)  # (order_id, product_name, quantity, total_cost)

> **Seealso:**
> The PostgreSQL ORM supports all the same operations as the SQLite version: string slicing (`column[1:5]`), pattern matching (`.startswith()`, `.endswith()`, `.contains()`), bulk operations with custom placeholders, and many more. Explore the API documentation for full details.